# Notebook 03: Interacting with the Blockchain using Web3.py

Smart contracts run on the blockchain, but we need a way to communicate with them from our Python applications.

Enter **Web3.py**! It is a Python library for interacting with Ethereum (and Ethereum-compatible) blockchains.

In this notebook, we will cover:
1. Connecting to a node.
2. Querying data (like balances).
3. Creating and sending transactions.
4. Interacting with deployed contracts.

### Prerequisites
We need to install `web3`. Since we are running in Colab and don't have a live node (like Geth or Ganache) running in the background, we will install the `eth-tester` plugin. This gives Web3.py a simulated, in-memory blockchain to talk to.

In [ ]:
!pip install "web3[tester]"

## 1. Connecting to Web3

Normally, you'd connect to a remote node (like Alchemy/Infura) or a local node via HTTP. Here, we use `EthereumTesterProvider`.

In [ ]:
from web3 import Web3

# Connect to the simulated test provider
w3 = Web3(Web3.EthereumTesterProvider())

print(f"Are we connected? {w3.is_connected()}")

# The test provider gives us 10 pre-funded accounts
accounts = w3.eth.accounts
print("Available test accounts:\n", accounts)

## 2. Querying Balances

Blockchain values are usually represented in `Wei` (the smallest unit of Ether). 
1 Ether = $10^{18}$ Wei.

In [ ]:
account_1 = accounts[0]
account_2 = accounts[1]

# Get balance in Wei
balance_wei = w3.eth.get_balance(account_1)
print(f"Balance of Account 1 (Wei): {balance_wei}")

# Convert to Ether for readability
balance_eth = w3.from_wei(balance_wei, 'ether')
print(f"Balance of Account 1 (Ether): {balance_eth} ETH")

## 3. Sending a Transaction

Let's send 5 Ether from Account 1 to Account 2. 

*Note: In `eth-tester`, transactions from the default accounts are often auto-signed, but in the real world, you must build the dictionary, sign it with your private key, and send the raw bytes.*

In [ ]:
# The amount to send, converted to Wei
amount_to_send = w3.to_wei(5, 'ether')

# Build the transaction dictionary
transaction = {
    'from': account_1,
    'to': account_2,
    'value': amount_to_send,
    'gas': 21000, # Standard gas limit for a simple ETH transfer
    'gasPrice': w3.to_wei('50', 'gwei'),
    'nonce': w3.eth.get_transaction_count(account_1),
}

# In the test environment, we can send it directly since Account 1 is unlocked
# In reality, you'd do: signed_txn = w3.eth.account.sign_transaction(transaction, private_key)
# and then: w3.eth.send_raw_transaction(signed_txn.rawTransaction)

print("Sending transaction...")
tx_hash = w3.eth.send_transaction(transaction)

print(f"Transaction Hash: {tx_hash.hex()}")

# Wait for the transaction to be mined (instant in test env)
receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
print("Transaction successful!")

# Check balances again
print(f"New Balance Acc 1: {w3.from_wei(w3.eth.get_balance(account_1), 'ether')} ETH")
print(f"New Balance Acc 2: {w3.from_wei(w3.eth.get_balance(account_2), 'ether')} ETH")